# 🚆 Mission SNCF : Les Trains de Retard
**Binôme : _______________ et _______________**

Bienvenue dans la peau d'une data analyst ! Vous allez enquêter sur les retards des trains Paris – Lyon pendant 2 semaines.

👉 Pour chaque cellule de code : cliquez dessus puis sur le bouton ▶ (à gauche).
👉 Les `____` sont les trous à compléter. Le reste du code est déjà écrit pour vous.
👉 Bloquées plus de 2 minutes ? Levez la main, une "cheffe de gare" arrive !

## 📦 Étape 0 — Charger les outils
On importe pandas (la bibliothèque magique des data analysts) et notre fichier.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "COLLEZ_ICI_LE_LIEN_VERS_TRAINS_RETARD_CSV"  # lien fourni par les organisatrices
trains = pd.read_csv(url)
trains  # on affiche le tableau

## 🔍 Étape 1 — Chasse au piège : les données sales !
**Question n°1 : ce tableau contient-il des erreurs ? Lesquelles ?**

Indice : regardez les valeurs les plus grandes et les cases vides...

In [ ]:
# Affiche les 10 plus gros retards à l'arrivée. Une valeur vous choque ?
trains.sort_values("Retard_arrivee_min", ascending=False).head(10)

In [ ]:
# Compte les cases vides dans chaque colonne
trains.isnull().____()

🎯 **Ce que vous devez trouver** :
1. Un retard de **999 minutes** : erreur de saisie (16h de retard, impossible !)
2. Une case **vide** dans Retard_depart_min (le 2026-03-05)
3. Une heure écrite **"7h15"** au lieu de "07:15" (le 2026-03-08)

Bravo, vous venez de faire du **nettoyage de données** : 80 % du boulot d'un data analyst !

In [ ]:
# ON CORRIGE ! On remplace le 999 par la moyenne des vrais retards
moyenne_arrivee = trains[trains["Retard_arrivee_min"] < 100]["Retard_arrivee_min"].mean()
trains["Retard_arrivee_min"] = trains["Retard_arrivee_min"].replace(999, int(moyenne_arrivee))

# On remplit la case vide avec la moyenne aussi
trains["Retard_depart_min"] = trains["Retard_depart_min"].fillna(trains["Retard_depart_min"].____())

# On remet l'heure au bon format
trains["Heure_depart"] = trains["Heure_depart"].replace("7h15", "07:15")

trains.sort_values("Retard_arrivee_min", ascending=False).head(3)

## 🕵️ Étape 2 — L'enquête commence !
### Question n°2 : Quel est le retard moyen des trains ? Et le pire trajet ?

In [ ]:
print("Retard moyen à l'arrivée :", trains["Retard_arrivee_min"].____(), "minutes")
print("Le trajet le PIRE :")
trains.loc[trains["Retard_arrivee_min"].idxmax()]

### Question n°3 : Est-ce que le type de train change quelque chose ?
(TGV vs TER vs Intercités)

In [ ]:
trains.groupby("____")["Retard_arrivee_min"].mean()

### Question n°4 : Est-ce que la MÉTÉO influence les retards ? 🌧️
Comparez le retard moyen par météo avec un graphique en barres.

In [ ]:
par_meteo = trains.groupby("Meteo")["Retard_arrivee_min"].mean()
par_meteo.plot(kind="bar", color="skyblue", title="Retard moyen selon la météo")
plt.ylabel("Retard à l'arrivée (min)")
plt.show()

# >> Que remarquez-vous ? Écrivez votre conclusion :
# "Quand il _____ ou _____, les trains sont en retard de ____ minutes en plus."

### Question n°5 : Est-ce que l'HEURE de départ influence les retards ? ⏰
Un petit calcul : extrayons l'heure (08:05 -> 8) pour comparer.

In [ ]:
trains["Heure"] = trains["Heure_depart"].str.split(":").str[0].astype(int)

par_heure = trains.groupby("____")["Retard_arrivee_min"].mean()
par_heure.plot(kind="bar", color="salmon", title="Retard moyen selon l'heure de départ")
plt.ylabel("Retard à l'arrivée (min)")
plt.show()

# >> Quelles heures sont les pires ? Pourquoi à votre avis ?
# (indice : que font les gens à 8h et à 18h ?)

### Question n°6 — LA grande découverte 🔥
**Est-ce qu'un train qui part en retard rattrape son retard ?**
Un scatterplot (nuage de points) va nous le révéler.

In [ ]:
trains.plot(kind="scatter", x="Retard_depart_min", y="Retard_arrivee_min",
            title="Retard au départ vs retard à l'arrivée")
plt.show()

# >> Les points forment-ils une ligne qui monte ? Que veut dire ça ?
# Conclusion : le retard au départ est presque toujours ______ à l'arrivée.

## 🎤 Étape 3 — Votre recommandation à la SNCF
**Question n°7 : que devrait faire la SNCF pour réduire les retards ?**
Complétez et cochez ce que vos données prouvent :

In [ ]:
recommandation = """
Chère SNCF, nos données de 2 semaines montrent que :
1. Les retards sont les plus forts quand il fait mauvais temps (facteur x1,5) : [VRAI / FAUX]
2. Les heures de pointe (8h, 17h-18h) sont les pires moments : [VRAI / FAUX]
3. Le retard se prend au DÉPART et n'est jamais rattrapé : [VRAI / FAUX]

Donc notre recommandation n°1 est : ________________________________
"""
print(recommandation)

## 🔮 Bonus (les plus rapides) — La prédiction
**Question n°8 : demain il pleut, le train part à 8h05. Pari : quel retard ?**
Une mini "intelligence artificielle" en 2 lignes (régression linéaire) :

In [ ]:
from sklearn.linear_model import LinearRegression

modele = LinearRegression()
modele.fit(trains[["Retard_depart_min"]], trains["Retard_arrivee_min"])

# Un train qui part avec 15 min de retard, qu'imagine l'IA ?
prediction = modele.predict([[15]])
print("L'IA prédit un retard à l'arrivée de :", round(float(prediction[0]), 1), "minutes")

# >> Vous venez d'entraîner un modèle de MACHINE LEARNING. Oui, juste là. 💅

## 🏁 Débrief métiers
Ce que vous venez de faire porte des noms :
- Nettoyer des données → **data cleaning**
- `groupby().mean()` → c'est une **agrégation** (comme en SQL)
- Les graphiques → **data visualisation**
- `modele.fit()` → **machine learning**

Tous ces gestes = le quotidien d'une data analyst. Bienvenue dans le métier ! 🚀